# Gerar Dados — Orquestrador

Ponto de entrada único que gera os dados sintéticos dos 4 sistemas (`SimuladorCRM`, `SimuladorERP`, `SimuladorTMS`, `SimuladorFinanceiro`) para uma data de referência, na ordem que respeita a cadeia de dependência (ADR-011): CRM -> ERP -> TMS -> Financeiro.

Parametrizado por `Widgets` (ADR-008), permitindo tanto a execução diária agendada quanto o reprocessamento manual de um dia/sistema específico, sem duplicar código.

Referências: ADR-001 (Landing Zone), ADR-003 (OOP/`SimuladorFactory`), ADR-008 (Widgets), ADR-010 (estratégia de geração), ADR-011 (ordem).

In [0]:
# instalações e importações 

%pip install dbldatagen Faker

In [0]:
dbutils.library.restartPython()

## Widgets

- `data_referencia`: data a processar (AAAA-MM-DD). Vazio = hoje.
- `sistema`: `"todos"` (ordem completa) ou um sistema específico, para reprocessamento isolado.
- `modo_execucao`: `"agendado"` (execução normal) ou `"reprocessamento_manual"` (registrado no resultado, para diferenciar depois na observabilidade).

In [0]:
dbutils.widgets.text("data_referencia", "", "Data de referência (AAAA-MM-DD)")
dbutils.widgets.dropdown("sistema", "todos", ["todos", "crm", "erp", "tms", "financeiro"], "Sistema")
dbutils.widgets.dropdown("modo_execucao", "agendado", ["agendado", "reprocessamento_manual"], "Modo de execução")

In [0]:
from datetime import date

data_referencia_str = dbutils.widgets.get("data_referencia")
data_referencia = date.fromisoformat(data_referencia_str) if data_referencia_str else date.today()

sistema_selecionado = dbutils.widgets.get("sistema")
modo_execucao = dbutils.widgets.get("modo_execucao")

print(f"data_referencia: {data_referencia}")
print(f"sistema: {sistema_selecionado}")
print(f"modo_execucao: {modo_execucao}")

## Execução

Se `sistema="todos"`, roda os 4 na ordem de dependência. Se um sistema específico for escolhido, roda só ele — útil para reprocessamento isolado (ex.: só o TMS teve problema num dia específico).

`gerar_seed()` é chamado a cada execução por simplicidade (é idempotente — sobrescreve o mesmo catálogo fixo — e barato); revisitar se isso se tornar custoso conforme o projeto evoluir.

Tratamento de erro/log estruturado em `pipeline_runs` (observabilidade) e notificação de falha (ADR-007) ainda não implementados — este notebook apenas gera e imprime o resultado por enquanto.

In [0]:
from src.simuladores.simulador_factory import SimuladorFactory

sistemas_a_rodar = (
    SimuladorFactory.ordem_execucao() if sistema_selecionado == "todos" else [sistema_selecionado]
)

resultados = []
for nome_sistema in sistemas_a_rodar:
    simulador = SimuladorFactory.criar(nome_sistema, spark=spark, dbutils=dbutils)
    simulador.executar_seed()
    resultado = simulador.gerar_dia(data_referencia)
    resultado["modo_execucao"] = modo_execucao
    resultados.append(resultado)
    print(resultado)

print("\nResumo da execução:")
for r in resultados:
    print(f"  {r['sistema']}: {r['status']} — {r.get('tabelas_geradas', [])}")